#### Uniform V2R re-count across all 12 genomes

initial CAFE said hydrophis # v2rs >>> any other species (which was known) ... hydrophis = I curated / NCBI pipeline for others ... therefore: here -> counted the V2Rs the same way for all genomes ... -> v2r reference library (v2r_library_hydrophis_all.fasta) -> scan 12 genomes w miniprot ... intact (>=80% ref coverage, no internal stops/frameshifts)

In [ ]:
CAFE = "/hpcfs/users/a1864358/sanders_lab/phylogenomics/cafe"
V2R = "/hpcfs/users/a1864358/sanders_lab/phylogenomics/cafe/v2r_recount"
counts = pd.read_csv(f"{V2R}/v2r_counts_uniform.tsv", sep="\t")
counts

,code,species,total_loci,intact,partial_pseudo
0,caspera,Candoia aspera,232,199,33
1,hcure,H. curtus E,776,663,113
2,hcurw,H. curtus W,1190,949,241
3,hcy,H. cyanocinctus,930,778,152
4,hmaj,H. major,1815,1537,278
5,horn,H. ornatus,867,696,171
6,nhel,Natrix helvetica,743,644,99
7,nscut,Notechis scutatus,352,234,118
8,ptext,Pseudonaja textilis,255,171,84
9,tele,Thamnophis elegans,349,278,71


#### Marine vs terrestrial-elapid vs other-outgroup means (uniform intact V2R count)

In [2]:
MARINE = ["hcy","hcure","hcurw","horn","hmaj"]
TERRESTRIAL_ELAPID = ["nscut","ptext"]
OTHER_OUTGROUP = ["caspera","vberus","nhel","tele","vkomodo"]
def grp(c):
    return "marine" if c in MARINE else "terrestrial_elapid" if c in TERRESTRIAL_ELAPID else "other_outgroup"
counts["group"] = counts.code.map(grp)
group_means = counts.groupby("group").intact.agg(["mean","min","max","count"]).round(2)
group_means

,mean,min,max,count
group,,,,
marine,924.6,663,1537,5
other_outgroup,308.4,157,644,5
terrestrial_elapid,202.5,171,234,2


In [3]:
marine_mean = counts.loc[counts.group=="marine","intact"].mean()
terr_mean = counts.loc[counts.group=="terrestrial_elapid","intact"].mean()
other_mean = counts.loc[counts.group=="other_outgroup","intact"].mean()
print(f"marine mean intact V2R = {marine_mean:.1f}")
print(f"terrestrial-elapid mean intact V2R = {terr_mean:.1f}")
print(f"other-outgroup mean intact V2R = {other_mean:.1f}")
print(f"\nIs marine higher than terrestrial-elapid? {'YES' if marine_mean > terr_mean else 'NO'} "
      f"({marine_mean:.1f} vs {terr_mean:.1f}, {marine_mean/terr_mean:.2f}x)")
print(f"Is marine higher than other outgroups? {'YES' if marine_mean > other_mean else 'NO'} "
      f"({marine_mean:.1f} vs {other_mean:.1f}, {marine_mean/other_mean:.2f}x)")

marine mean intact V2R = 924.6
terrestrial-elapid mean intact V2R = 202.5
other-outgroup mean intact V2R = 308.4

Is marine higher than terrestrial-elapid? YES (924.6 vs 202.5, 4.57x)
Is marine higher than other outgroups? YES (924.6 vs 308.4, 3.00x)


##### Robustness check: does the OrthoFinder marine:terrestrial ratio match the uniform-recount ratio?

In [4]:
og_annot = pd.read_csv(f"{V2R}/../inputs/og_annotation.tsv", sep="\t")
v2r_ogs = og_annot[og_annot.consensus_product.str.contains("Vomeronasal type-2", na=False)].FamilyID
print("V2R orthogroups (OrthoFinder, Hydrophis-annotation-derived label):", len(v2r_ogs))

cmp_df = pd.read_csv(f"{V2R}/v2r_counts_comparison.tsv", sep="\t")
cmp_df["group"] = cmp_df.code.map(grp)
cmp_df

V2R orthogroups (OrthoFinder, Hydrophis-annotation-derived label): 325


,code,species,orthofinder_v2r,uniform_intact_v2r,group
0,caspera,Candoia aspera,591,199,other_outgroup
1,hcure,H. curtus E,854,663,marine
2,hcurw,H. curtus W,912,949,marine
3,hcy,H. cyanocinctus,828,778,marine
4,hmaj,H. major,1064,1537,marine
5,horn,H. ornatus,841,696,marine
6,nhel,Natrix helvetica,517,644,other_outgroup
7,nscut,Notechis scutatus,371,234,terrestrial_elapid
8,ptext,Pseudonaja textilis,406,171,terrestrial_elapid
9,tele,Thamnophis elegans,636,278,other_outgroup


In [ ]:
of_marine_mean = cmp_df.loc[cmp_df.group=="marine","orthofinder_v2r"].mean()
of_terr_mean = cmp_df.loc[cmp_df.group=="terrestrial_elapid","orthofinder_v2r"].mean()
of_ratio = of_marine_mean/of_terr_mean
uni_ratio = marine_mean/terr_mean
print(f"OrthoFinder-based:      marine mean={of_marine_mean:.1f}  terrestrial mean={of_terr_mean:.1f}  ratio={of_ratio:.2f}x")
print(f"Uniform-recount-based:  marine mean={marine_mean:.1f}  terrestrial mean={terr_mean:.1f}  ratio={uni_ratio:.2f}x")

"""
Results:
OrthoFinder-based:      marine mean=899.8  terrestrial mean=388.5  ratio=2.32x
Uniform-recount-based:  marine mean=924.6  terrestrial mean=202.5  ratio=4.57x

"""

## conclusion

Hydrophis V2Rs were aligned and scanned across the species panel. Hydrophis recorded 4.6× more hits than terrestrial elapids (mean 924.6 vs 202.5), and 3.0× more than the further outgroups (mean 924.6 vs 308.4). This demonstrates Hydrophis V2Rs have diverged from terrestrial transcripts, as alignment coverage against the Hydrophis-derived reference drops off with phylogenetic distance, and the scale of the drop-off — far larger toward elapids than sequence divergence alone would predict at this evolutionary distance — supports a marine expansion as well.